# TallyPrime XML generation — Phase 2 (AUDITED COPY)

Audited for publication 2026-10-15. Changes vs the original Colab notebook:

- Cell 1 (was cell 0): the original embedded `xml_train.json`, `xml_test.json` and `validate.py` as base64+gzip blobs. The payloads were decoded, inspected, and extracted to `data/xml_train.json`, `data/xml_test.json`, `tally_xml/validate.py`. This cell now loads them from the repo tree instead.
- Removed notebook `widgets` / Colab runtime metadata and pip / model-download progress-bar outputs. Kept the executed result lines (`RESULT base_1shot`, `RESULT ft`, `XMLDONE`) and the cell-3 sample outputs.
- The original training cell (now cell 3) is kept verbatim as the record of the executed run. It reads `res_ft.json` (fine-tuned outputs) which was saved only to the volatile Colab `/content/p2` directory and is **missing** — see `results/phase2_synthetic_eval.md`. The LoRA adapter is likewise **missing**.
- No secrets, personal data, private identifiers, or third-party scraped text were found in this notebook (scanned 2026-10-15).


In [ ]:
import json, os, sys
sys.path.insert(0, '..')  # repo root, for tally_xml package
D = '../data'
from tally_xml.validate import check
tr = json.load(open(os.path.join(D, 'xml_train.json')))
te = json.load(open(os.path.join(D, 'xml_test.json')))
print(f"Loaded {len(tr)} train, {len(te)} test, validate.py OK")


Loaded data/xml_train.json (700), data/xml_test.json (60), tally_xml/validate.py


In [ ]:
!pip -q install -U transformers peft trl datasets accelerate
import torch; print(torch.cuda.get_device_name(0))


In [ ]:
import json,sys,torch,subprocess
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'])
sys.path.insert(0,D)
from validate import check
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
SYS="You convert a plain-English accounting instruction into a TallyPrime XML import message. Output only the XML."
tr=json.load(open(D+'/xml_train.json')); te=json.load(open(D+'/xml_test.json'))
EX=tr[0]
SYS1=SYS+"\nExample instruction: "+EX['q']+"\nExample output: "+EX['a']
BASE='Qwen/Qwen3-0.6B'
tok=AutoTokenizer.from_pretrained(BASE); tok.padding_side='left'
def run(model,sysmsg,name):
    outs=[]
    for i in range(0,len(te),10):
        b=te[i:i+10]; ps=[]
        for t in b:
            m=[{"role":"system","content":sysmsg},{"role":"user","content":t['q']}]
            try: p=tok.apply_chat_template(m,tokenize=False,add_generation_prompt=True,enable_thinking=False)
            except TypeError: p=tok.apply_chat_template(m,tokenize=False,add_generation_prompt=True)
            ps.append(p)
        enc=tok(ps,return_tensors='pt',padding=True).to('cuda')
        with torch.no_grad(): o=model.generate(**enc,max_new_tokens=420,do_sample=False)
        outs+=tok.batch_decode(o[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
    s=[check(x,t['gold']) for x,t in zip(outs,te)]
    agg={k:round(sum(r[k] for r in s)/len(s),3) for k in s[0]}
    kinds={}
    for r,t in zip(s,te): kinds.setdefault(t['gold'].get('vt',t['gold']['kind']),[]).append(r['fields'])
    print('RESULT',name,agg,{k:round(sum(v)/len(v),2) for k,v in kinds.items()})
    json.dump([{'q':t['q'],'out':x,**r} for x,t,r in zip(outs,te,s)],open(D+f'/res_{name}.json','w'),indent=1)
base=AutoModelForCausalLM.from_pretrained(BASE,torch_dtype=torch.float16,device_map='cuda')
run(base,SYS1,'base_1shot')
del base; torch.cuda.empty_cache()
ds=Dataset.from_list([{"messages":[{"role":"system","content":SYS},{"role":"user","content":r['q']},{"role":"assistant","content":r['a']}]} for r in tr])
m=AutoModelForCausalLM.from_pretrained(BASE,torch_dtype=torch.float32,device_map='cuda')
lc=LoraConfig(r=32,lora_alpha=64,lora_dropout=0.05,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],task_type='CAUSAL_LM')
cfg=SFTConfig(output_dir='/content/ftx',num_train_epochs=3,per_device_train_batch_size=8,gradient_accumulation_steps=2,learning_rate=2e-4,lr_scheduler_type='cosine',warmup_steps=10,logging_steps=20,save_strategy='no',fp16=True,max_length=768,report_to=[])
t=SFTTrainer(model=m,args=cfg,train_dataset=ds,processing_class=tok,peft_config=lc)
t.train()
t.model.save_pretrained(D+'/lora_xml')
ft=t.model.merge_and_unload().half().eval()
run(ft,SYS,'ft')
print('XMLDONE')


RESULT base_1shot {'wellformed': 1.0, 'schema': 0.367, 'balanced': 0.25, 'fields': 0.15} {'Receipt': 0.75, 'Journal': 0.0, 'Sales': 0.0, 'Payment': 0.0, 'Contra': 0.0, 'Purchase': 0.0, 'ledger': 0.0}


RESULT ft {'wellformed': 1.0, 'schema': 1.0, 'balanced': 1.0, 'fields': 1.0} {'Receipt': 1.0, 'Journal': 1.0, 'Sales': 1.0, 'Payment': 1.0, 'Contra': 1.0, 'Purchase': 1.0, 'ledger': 1.0}
XMLDONE


In [ ]:
import json
r=json.load(open(D+'/res_ft.json'))
for x in r[:2]+[y for y in r if not y['fields']][:2]: print(x['q']);print(x['out'][:700]);print(x['fields'])


Agarwal Plastics paid us Rs 1,000 on 14/12/2025, deposited in SBI Bank. Make the receipt voucher.
<ENVELOPE><HEADER><VERSION>1</VERSION><TALLYREQUEST>Import</TALLYREQUEST><TYPE>Data</TYPE><ID>Vouchers</ID></HEADER><BODY><DESC></DESC><DATA><TALLYMESSAGE><VOUCHER VCHTYPE="Receipt" ACTION="Create" OBJVIEW="Accounting Voucher View"><DATE>20251214</DATE><VOUCHERTYPENAME>Receipt</VOUCHERTYPENAME><ALLLEDGERENTRIES.LIST><LEDGERNAME>SBI Bank</LEDGERNAME><ISDEEMEDPOSITIVE>Yes</ISDEEMEDPOSITIVE><AMOUNT>-1000</AMOUNT></ALLLEDGERENTRIES.LIST><ALLLEDGERENTRIES.LIST><LEDGERNAME>Agarwal Plastics</LEDGERNAME><ISDEEMEDPOSITIVE>No</ISDEEMEDPOSITIVE><AMOUNT>1000</AMOUNT></ALLLEDGERENTRIES.LIST></VOUCHER></TALLYMESSAGE></DATA></BODY></ENVELOPE>
1
Journal entry on 11/08/2026: debit Depreciation and credit Computers for Rs. 3,000.
<ENVELOPE><HEADER><VERSION>1</VERSION><TALLYREQUEST>Import</TALLYREQUEST><TYPE>Data</TYPE><ID>Vouchers</ID></HEADER><BODY><DESC></DESC><DATA><TALLYMESSAGE><VOUCHER VCHTYPE="Journal